# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm

## 1. Preprocessing pipeline

In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['weekday'] = df['timestamp'].dt.dayofweek
        df = df.drop(columns=['timestamp'])
        return df

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.encoder = OneHotEncoder(sparse_output=False)
        self.cat_cols = None
    
    def fit(self, X, y=None):
        df = X.copy()
        self.cat_cols = [c for c in df.select_dtypes(include='object').columns
                         if c != self.target_col]
        self.encoder.fit(df[self.cat_cols])
        return self
    
    def transform(self, X):
        df = X.copy()
        encoded = self.encoder.transform(df[self.cat_cols])
        enc_cols = self.encoder.get_feature_names_out(self.cat_cols)
        enc_df = pd.DataFrame(encoded, columns=enc_cols, index=df.index)
        df = df.drop(columns=self.cat_cols)
        df = pd.concat([df, enc_df], axis=1)
        target = df[self.target_col]
        X_out = df.drop(columns=[self.target_col])
        return X_out, target

In [4]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state
    
    def split(self, X, y):
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, test_size=self.test_size,
            random_state=self.random_state, stratify=y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_trainval, y_trainval, test_size=self.test_size,
            random_state=self.random_state, stratify=y_trainval
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

In [5]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = {}
    
    def choose(self, X_train, y_train, X_valid, y_valid):
        best_name = None
        best_score = 0
        
        for i, gs in enumerate(self.grids):
            name = self.grid_dict[i]
            print(f'Estimator: {name}')
            
            for _ in tqdm(range(1)):
                gs.fit(X_train, y_train)
            
            print(f'Best params: {gs.best_params_}')
            print(f'Best training accuracy: {gs.best_score_:.3f}')
            
            val_score = accuracy_score(y_valid, gs.best_estimator_.predict(X_valid))
            print(f'Validation set accuracy score for best params: {val_score:.3f} \n')
            
            self.best_models[name] = {
                'model': gs.best_estimator_,
                'params': gs.best_params_,
                'valid_score': val_score
            }
            
            if val_score > best_score:
                best_score = val_score
                best_name = name
        
        print(f'Classifier with best validation set accuracy: {best_name}')
        return best_name
    
    def best_results(self):
        rows = []
        for name, info in self.best_models.items():
            rows.append({
                'model': name,
                'params': info['params'],
                'valid_score': info['valid_score']
            })
        return pd.DataFrame(rows)

## 3. Finalization

In [6]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
    
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        acc = accuracy_score(y_test, self.estimator.predict(X_test))
        print(f'Accuracy of the final model is {acc}')
        return acc
    
    def save_model(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self.estimator, f)
        print(f'Model successfully saved to {path}')

## 4. Main program

In [7]:
# Load data
df = pd.read_csv('../data/checker_submits.csv')
df.head()

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


In [8]:
# Preprocessing pipeline
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])

data = preprocessing.fit_transform(df)
X_out, y_out = data
print('X shape:', X_out.shape)
print('y shape:', y_out.shape)

X shape: (1686, 43)
y shape: (1686,)


In [9]:
# Train/valid/test split
splitter = TrainValidationTest(test_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(
    X_out.values, y_out.values
)
print('Train:', X_train.shape)
print('Valid:', X_valid.shape)
print('Test:', X_test.shape)

Train: (1078, 43)
Valid: (270, 43)
Test: (338, 43)


In [10]:
# Define gridsearch objects
svm_params = [{
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}]

tree_params = [{
    'max_depth': list(range(1, 30)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}]

rf_params = [{
    'n_estimators': [5, 10, 50],
    'max_depth': list(range(1, 30)),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}]

gs_svm = GridSearchCV(SVC(), svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(DecisionTreeClassifier(), tree_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(RandomForestClassifier(), rf_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

In [11]:
# Model selection
ms = ModelSelection(grids, grid_dict)
best_name = ms.choose(X_train, y_train, X_valid, y_valid)
print(ms.best_results())

Estimator: SVM


  0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree


  0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22, 'random_state': 21}
Best training accuracy: 0.809
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest


  0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.904 

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': 'balanced', 'criterion': 'gin...   
2  Random Forest  {'class_weight': None, 'criterion': 'gini', 'm...   

   valid_score  
0     0.877778  
1     0.866667  
2     0.903704  


In [12]:
# Finalize best model
best_estimator = ms.best_models[best_name]['model']
final = Finalize(best_estimator)
acc = final.final_score(X_train, y_train, X_test, y_test)
model_name = f'{best_name.replace(" ", "_")}_{acc:.2f}.sav'
final.save_model(f'../data/{model_name}')

Accuracy of the final model is 0.9112426035502958
Model successfully saved to ../data/Random_Forest_0.91.sav
